In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D9 — Movimento Fisiológico da População de Portugal, 1925
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!apt-get update -qq
!apt-get install -y poppler-utils -qq
!pip install -q pymupdf pdf2image

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import math
import platform
import sys

import fitz
import pandas as pd

from pdf2image import convert_from_path

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D9"

DOCUMENT_NAME = (
    "Estatística do Movimento Fisiológico da População "
    "de Portugal — Ano de 1925"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".pdf"

INPUT_REPRESENTATION = "Original scanned PDF"

DIRECT_DOCUMENT_INGESTION = True

EXPECTED_PAGE_COUNT = 8


# ------------------------------------------------------------
# Fixed Stage 1 reference expectations
#
# Notebook-side diagnostics only.
# These values must NOT be supplied to the model.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 19

EXPECTED_CATEGORY_COUNTS = {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3
}


# ------------------------------------------------------------
# Fixed Stage 1 extraction schema
# ------------------------------------------------------------

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]


STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


VALUE_ALLOWED_TYPES = (
    str,
    int,
    float,
    type(None)
)


# Content-completeness diagnostic only.
# This is separate from technical field-type validity.
MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location"
]


ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS
)


# ------------------------------------------------------------
# Fixed Stage 1 record topics
#
# These identify the predefined task scope.
# Values remain undisclosed to the model.
# ------------------------------------------------------------

PUBLICATION_METADATA_TOPICS = [
    "Title",
    "Reference year",
    "Publication year",
    "Publisher",
    "Institution"
]


INDEX_ENTRY_TOPICS = [
    "Tabela I",
    "Tabela II",
    "Tabela III",
    "Tabela XIV",
    "Tabela LVIII",
    "Tabela LIX"
]


STATISTICAL_TOPICS = [
    "Portugal area",
    "Portugal population 1911",
    "Portugal population 1920",
    "Portugal density 1920",
    "Portugal average annual population growth"
]


STRUCTURAL_TOPICS = [
    "Rotated table",
    "Bilingual headings",
    "Historical typography"
]


# ------------------------------------------------------------
# Stage 1 reference values used only AFTER extraction
# ------------------------------------------------------------

EXPECTED_STATISTICAL_VALUES = {
    "Portugal area": 91948.07,
    "Portugal population 1911": 5960056,
    "Portugal population 1920": 6032991,
    "Portugal density 1920": 65.6,
    "Portugal average annual population growth": 1.36
}


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs_D9_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D9_branch_A_input_integrity.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D9_branch_A_representation.json"
)

PROMPT_PATH = (
    OUTPUT_DIR
    / "D9_branch_A_prompt.txt"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D9_branch_A_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D9_branch_A_parsed_extraction.json"
)

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D9_branch_A_technical_diagnostics.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D9_branch_A_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D9_branch_A_experiment_summary.json"
)


print(
    "Document:",
    DOCUMENT_ID
)

print(
    "Branch:",
    BRANCH
)

print(
    "Input representation:",
    INPUT_REPRESENTATION
)

print(
    "Expected pages:",
    EXPECTED_PAGE_COUNT
)

print(
    "Expected reference records:",
    EXPECTED_RECORD_COUNT
)

print(
    "Expected fields:",
    len(EXPECTED_FIELDS)
)

In [ ]:
# ============================================================
# 2. Source document and integrity diagnostics
# ============================================================

print(
    "Upload the original D9 scanned PDF."
)


uploaded = files.upload()


pdf_paths = [
    Path(name)

    for name
    in uploaded

    if name.lower().endswith(
        ".pdf"
    )
]


if len(pdf_paths) != 1:

    raise ValueError(
        "Upload exactly one PDF source document."
    )


SOURCE_PATH = pdf_paths[0]


# ------------------------------------------------------------
# File hashing
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)


FILE_SIZE_BYTES = (
    SOURCE_PATH.stat().st_size
)

FILE_NON_EMPTY = (
    FILE_SIZE_BYTES > 0
)


# ------------------------------------------------------------
# Inspect PDF structure and native text layer
# ------------------------------------------------------------

pdf_document = fitz.open(
    SOURCE_PATH
)


PAGE_COUNT = len(
    pdf_document
)


PAGE_COUNT_VALID = (
    PAGE_COUNT
    == EXPECTED_PAGE_COUNT
)


page_rows = []


for page_number, page in enumerate(
    pdf_document,
    start=1
):

    native_text = (
        page.get_text(
            "text"
        )
        or ""
    )


    page_rows.append(
        {
            "Page Number":
                page_number,

            "Native Character Count":
                len(
                    native_text
                ),

            "Native Word Count":
                len(
                    native_text.split()
                ),

            "Native Text Extractable":
                bool(
                    native_text.strip()
                ),

            "Width":
                float(
                    page.rect.width
                ),

            "Height":
                float(
                    page.rect.height
                ),

            "PDF Rotation":
                int(
                    page.rotation
                )
        }
    )


page_diagnostics_df = pd.DataFrame(
    page_rows
)


TOTAL_NATIVE_CHARACTERS = int(
    page_diagnostics_df[
        "Native Character Count"
    ].sum()
)


TEXT_EXTRACTABLE = bool(
    TOTAL_NATIVE_CHARACTERS > 100
)


IMAGE_BASED_SOURCE = (
    not TEXT_EXTRACTABLE
)


OCR_DEPENDENCY_FOR_TEXT_ONLY_PROCESSING = (
    IMAGE_BASED_SOURCE
)


# ------------------------------------------------------------
# Render pages for diagnostic metadata only
# ------------------------------------------------------------

RENDER_DPI = 150


rendered_pages = convert_from_path(
    str(
        SOURCE_PATH
    ),
    dpi=RENDER_DPI,
    fmt="png"
)


RENDERED_PAGE_COUNT = len(
    rendered_pages
)


RENDERED_PAGE_COUNT_VALID = (
    RENDERED_PAGE_COUNT
    == PAGE_COUNT
)


image_rows = []


for page_number, image in enumerate(
    rendered_pages,
    start=1
):

    orientation = (
        "landscape"
        if image.width > image.height
        else "portrait"
    )


    image_rows.append(
        {
            "Page Number":
                page_number,

            "Image Width":
                image.width,

            "Image Height":
                image.height,

            "Orientation":
                orientation,

            "Rendered DPI":
                RENDER_DPI
        }
    )


image_diagnostics_df = pd.DataFrame(
    image_rows
)


# ------------------------------------------------------------
# D9 source characteristics
# ------------------------------------------------------------

STAGE1_VISUAL_CHARACTERISTICS = {
    "contains_cover_page":
        True,

    "contains_title_page":
        True,

    "contains_index_pages":
        True,

    "contains_historical_tables":
        True,

    "contains_rotated_table_content":
        True,

    "rotated_content_physical_pages":
        [7, 8],

    "contains_bilingual_portuguese_french_headings":
        True,

    "contains_historical_typography":
        True,

    "contains_dense_numeric_tables":
        True
}


# ------------------------------------------------------------
# Input-integrity status
# ------------------------------------------------------------

DIRECT_PDF_INGESTION_USABLE = all([
    FILE_NON_EMPTY,
    PAGE_COUNT_VALID,
    RENDERED_PAGE_COUNT_VALID
])


INPUT_INTEGRITY_PASSED = (
    DIRECT_PDF_INGESTION_USABLE
)


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "source_format":
        SOURCE_FORMAT,

    "file_size_bytes":
        FILE_SIZE_BYTES,

    "file_non_empty":
        FILE_NON_EMPTY,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "observed_page_count":
        PAGE_COUNT,

    "page_count_valid":
        PAGE_COUNT_VALID,

    "native_text_characters":
        TOTAL_NATIVE_CHARACTERS,

    "meaningful_native_text_layer":
        TEXT_EXTRACTABLE,

    "image_based_source":
        IMAGE_BASED_SOURCE,

    "ocr_dependency_for_text_only_processing":
        OCR_DEPENDENCY_FOR_TEXT_ONLY_PROCESSING,

    "ocr_applied_for_branch_a_model_input":
        False,

    "diagnostic_page_rendering_applied":
        True,

    "diagnostic_render_dpi":
        RENDER_DPI,

    "rendered_page_count":
        RENDERED_PAGE_COUNT,

    "rendered_page_count_valid":
        RENDERED_PAGE_COUNT_VALID,

    "rendered_images_used_as_model_input":
        False,

    "stage1_visual_characteristics":
        STAGE1_VISUAL_CHARACTERISTICS,

    "direct_pdf_ingestion_usable":
        DIRECT_PDF_INGESTION_USABLE,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED
}


INPUT_INTEGRITY_PATH.write_text(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "Source:",
    SOURCE_PATH.name
)

print(
    "Source size:",
    f"{FILE_SIZE_BYTES:,} bytes"
)

print(
    "SHA-256:",
    SOURCE_SHA256
)

print(
    "Observed pages:",
    PAGE_COUNT
)

print(
    "Page count valid:",
    PAGE_COUNT_VALID
)

print(
    "Native characters:",
    TOTAL_NATIVE_CHARACTERS
)

print(
    "Meaningful native text layer:",
    TEXT_EXTRACTABLE
)

print(
    "Image-based source:",
    IMAGE_BASED_SOURCE
)

print(
    "OCR dependency for text-only processing:",
    OCR_DEPENDENCY_FOR_TEXT_ONLY_PROCESSING
)

print(
    "Rendered pages:",
    RENDERED_PAGE_COUNT
)

print(
    "Input integrity passed:",
    INPUT_INTEGRITY_PASSED
)


display(
    page_diagnostics_df
)

display(
    image_diagnostics_df
)

if not FILE_NON_EMPTY:

    raise AssertionError(
        "The D9 source file is empty."
    )


if not PAGE_COUNT_VALID:

    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"found {PAGE_COUNT}."
    )


if not RENDERED_PAGE_COUNT_VALID:

    raise AssertionError(
        "Diagnostic rendered-page count does not "
        "match the PDF page count."
    )

In [ ]:
# ============================================================
# 3. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "source_representation":
        "Image-based historical scanned PDF",

    "complete_original_document_supplied":
        True,

    "direct_document_ingestion":
        True,

    "native_text_inspection_applied":
        True,

    "native_text_used_as_model_input":
        False,

    "diagnostic_page_rendering_applied":
        True,

    "diagnostic_render_dpi":
        RENDER_DPI,

    "rendered_images_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "ocr_transcript_used_as_model_input":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "page_extraction_applied":
        False,

    "page_cropping_applied":
        False,

    "page_rotation_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_reconstruction_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "model_input_description": (
        "The complete original eight-page D9 scanned PDF is "
        "submitted directly to the LLM. Native-text inspection "
        "and 150-DPI page rendering are used only for notebook "
        "diagnostics. No OCR transcript or rendered diagnostic "
        "image representation is supplied to the model."
    )
}


REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 4. Extraction prompt
# ============================================================

BRANCH_A_PROMPT = """You are an information extraction assistant.

Extract the predefined bibliographic, index, statistical and structural
records represented within the defined scope of the attached original
historical scanned PDF publication:

“Estatística do Movimento Fisiológico da População de Portugal —
Ano de 1925”.

Treat the attached original PDF as the only source of information.

The supplied PDF is a subset of a larger historical publication.
Use only the physical pages contained in the supplied PDF and do not
infer information from omitted printed pages.

Use exactly these record fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Use exactly one of these Category values:

- Publication metadata
- Index entry
- Statistical value
- Document structure


1. Publication metadata

Using the cover and imprint represented on physical PDF page 1,
extract one record for each of these predefined topics:

- Title
- Reference year
- Publication year
- Publisher
- Institution

Preserve the represented Portuguese title and institution wording.

Keep the statistical reference year distinct from the printed
publication year.

Do not infer bibliographic information that is not explicitly visible
in the supplied source.


2. Selected index entries

Using the index pages contained in the supplied PDF, extract one record
for each of these selected table identifiers:

- Tabela I
- Tabela II
- Tabela III
- Tabela XIV
- Tabela LVIII
- Tabela LIX

For each selected index entry:

- preserve the Portuguese table identifier;
- preserve the Portuguese table description;
- preserve the represented printed page number or page range in the
  Source Location;
- preserve any explicitly associated period;
- do not create a separate record from the accompanying French
  translation.

The first four selected entries are located on physical PDF page 3.
The final two selected entries are located on physical PDF page 5.


3. Selected statistical values from Tabela I

From the Portugal row of Tabela I on physical PDF page 6, extract one
record for each of these predefined indicators:

- Portugal area
- Portugal population 1911
- Portugal population 1920
- Portugal density 1920
- Portugal average annual population growth

Read each value directly from the represented Portugal row and its
corresponding column.

Preserve the printed numeric scale.

Do not calculate, derive, estimate, interpolate, rescale, correct or
convert any value.

Do not extract district-level, city-level, sex-specific or other
Tabela I observations outside this predefined scope.


4. Document structure

Create one source-grounded structural record for each of these
predefined topics:

- Rotated table
- Bilingual headings
- Historical typography

For “Rotated table”, describe the visual orientation and multi-page
presentation of Tabela II in the supplied source.

For “Bilingual headings”, describe the represented relationship between
the Portuguese headings and their accompanying translated headings.

For “Historical typography”, describe the visibly represented historical
typographic and scanned tabular presentation.

These structural observations must be grounded only in visible
properties of the supplied document.

Do not infer any quantitative values from Tabela II.


Field rules:

Category:
- Use exactly one of the four category labels defined above.

Topic:
- For each record, use the corresponding predefined topic label from
  the scope above.

Description:
- Provide a concise source-grounded description of the represented
  record.
- Preserve Portuguese source wording where the record is a publication
  title or table description.
- Do not add external interpretation.

Value:
- Use a JSON number when the source explicitly represents a numeric
  value.
- Use a JSON string when the represented value is textual.
- Use null only when no separate Value is represented.
- Preserve the represented printed scale.
- Do not calculate, infer, derive, convert, repair or modernise values.

Unit:
- Preserve the explicitly associated measurement unit where applicable.
- Use null when no explicit unit applies.
- Do not invent units.

Reporting Period:
- Preserve an explicitly associated reference year or period.
- Keep the statistical reference year and publication year distinct.
- Preserve the census periods associated with the selected statistical
  observations.
- Use null when no explicit reporting period applies.

Source Location:
- Use physical PDF page references grounded in the supplied file.
- Where relevant, also preserve the represented printed page or page
  range for an index entry.
- Use concise locations such as:
  “PDF page 1 — Cover”
  “PDF page 1 — Imprint”
  “PDF page 3 — Índice; printed page ...”
  “PDF page 5 — Índice; printed page ...”
  “PDF page 6 — Tabela I, Portugal row”
  “PDF pages 7–8 — Tabela II”


Additional extraction rules:

- Use only information explicitly represented in the supplied PDF.
- Preserve Portuguese titles, table identifiers, historical spelling
  and punctuation where relevant.
- Do not modernise or silently correct source wording.
- Do not treat French translations as separate duplicate records.
- Do not use OCR output supplied from another source.
- Do not use external knowledge.
- Do not calculate, infer, derive, estimate or reconstruct missing
  information.
- Do not infer age-by-sex or other quantitative observations from
  Tabela II.
- Do not extract observations outside the predefined scope.
- Verify that every item within the defined scope has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D9",
  "branch": "A",
  "records": [
    {
      "Category": null,
      "Topic": null,
      "Description": null,
      "Value": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_A_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)


print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

print()

print(
    BRANCH_A_PROMPT
)

## Independent Branch A extraction

Open a new independent conversation.

Upload:

1. the complete original D9 scanned PDF;
2. `D9_branch_A_prompt.txt`.

Submit the prompt once.

Save the complete, untouched model response as:

`D9_branch_A_raw_response.txt`


In [ ]:
# ============================================================
# 5. Raw response preservation and parsing
# ============================================================

print(
    "Upload the untouched "
    "D9_branch_A_raw_response.txt file."
)


uploaded = files.upload()


txt_paths = [
    Path(name)

    for name
    in uploaded

    if name.lower().endswith(
        ".txt"
    )
]


if len(txt_paths) != 1:

    raise ValueError(
        "Upload exactly one TXT raw-response file."
    )


UPLOADED_RAW_RESPONSE_PATH = (
    txt_paths[0]
)


raw_response_text = (
    UPLOADED_RAW_RESPONSE_PATH.read_text(
        encoding="utf-8"
    )
)


if not raw_response_text.strip():

    raise ValueError(
        "The uploaded raw response is empty."
    )


# ------------------------------------------------------------
# Preserve untouched raw response BEFORE parsing
# ------------------------------------------------------------

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


# ------------------------------------------------------------
# Parse without crashing when model JSON is invalid
# ------------------------------------------------------------

valid_json = False

json_parsing_error = None

parsed_response = None


try:

    parsed_response = json.loads(
        raw_response_text
    )

    valid_json = True


except json.JSONDecodeError as error:

    json_parsing_error = str(
        error
    )


# ------------------------------------------------------------
# Validate standardized top-level wrapper
# ------------------------------------------------------------

top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)


document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)


document_id_correct = (
    top_level_object_valid
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)


branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)


branch_correct = (
    top_level_object_valid
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)


records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)


records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)


# ------------------------------------------------------------
# Record-level diagnostics are possible whenever a valid
# ------------------------------------------------------------

records_evaluable = (
    valid_json
    and top_level_object_valid
    and records_present
    and records_is_list
)


if records_evaluable:

    extracted_records = (
        parsed_response[
            "records"
        ]
    )

    observed_record_count = len(
        extracted_records
    )


else:

    extracted_records = []

    observed_record_count = None


# ------------------------------------------------------------
# Create parsed extraction only when records are evaluable
# ------------------------------------------------------------

parsed_extraction_created = False

parsed_extraction_sha256 = None


if records_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get(
                "document_id"
            ),

        "branch":
            parsed_response.get(
                "branch"
            ),

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_created = True

    parsed_extraction_sha256 = (
        sha256_file(
            PARSED_EXTRACTION_PATH
        )
    )


print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed record count:",
    observed_record_count
)


if records_evaluable:

    extracted_df = pd.DataFrame(
        extracted_records
    )

    display(
        extracted_df
    )

In [ ]:
# ============================================================
# 6. Record and content diagnostics
# ============================================================

record_structure_issues = []

field_type_issues = []

missing_mandatory_values = []


# ------------------------------------------------------------
# A. Exact record schema
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Record is not a JSON object"
                }
            )

            continue


        observed_fields = list(
            record.keys()
        )


        if (
            observed_fields
            != EXPECTED_FIELDS
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        (
                            "Field names or field "
                            "order differ"
                        ),

                    "expected_fields":
                        EXPECTED_FIELDS,

                    "observed_fields":
                        observed_fields,

                    "missing_fields":
                        [
                            field

                            for field
                            in EXPECTED_FIELDS

                            if field
                            not in record
                        ],

                    "extra_fields":
                        [
                            field

                            for field
                            in observed_fields

                            if field
                            not in EXPECTED_FIELDS
                        ]
                }
            )


    records_with_structure_issues = len({
        issue[
            "record_index"
        ]

        for issue
        in record_structure_issues
    })


    record_schema_valid = (
        records_with_structure_issues
        == 0
    )


else:

    records_with_structure_issues = None

    record_schema_valid = None


# ------------------------------------------------------------
# B. Field types
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            continue


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )


            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__,

                        "expected_type":
                            "string or null"
                    }
                )


        value = record.get(
            "Value"
        )


        if (
            isinstance(
                value,
                bool
            )
            or not isinstance(
                value,
                VALUE_ALLOWED_TYPES
            )
        ):

            field_type_issues.append(
                {
                    "record_index":
                        record_index,

                    "field":
                        "Value",

                    "observed_type":
                        type(
                            value
                        ).__name__,

                    "expected_type":
                        "string, number or null"
                }
            )


        # ----------------------------------------------------
        # Mandatory-content diagnostic
        # ----------------------------------------------------

        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )


            if (
                value is None
                or value == ""
            ):

                missing_mandatory_values.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field
                    }
                )


    records_with_type_issues = len({
        issue[
            "record_index"
        ]

        for issue
        in field_type_issues
    })


    field_types_valid = (
        records_with_type_issues
        == 0
    )


    missing_mandatory_value_count = len(
        missing_mandatory_values
    )


    mandatory_fields_complete = (
        missing_mandatory_value_count
        == 0
    )


else:

    records_with_type_issues = None

    field_types_valid = None

    missing_mandatory_value_count = None

    mandatory_fields_complete = None


# ------------------------------------------------------------
# C. Record-count and category diagnostics
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )

            for record
            in extracted_records

            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


else:

    record_count_valid = None

    observed_category_counts = None

    categories_valid = None

    category_counts_valid = None


# ------------------------------------------------------------
# D. Complete-record duplicate diagnostic
# ------------------------------------------------------------

if records_evaluable:

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(
                    field
                ),
                ensure_ascii=False,
                sort_keys=True
            )

            for field
            in EXPECTED_FIELDS
        )

        for record
        in extracted_records

        if isinstance(
            record,
            dict
        )
    )


    duplicate_records = [
        list(
            key
        )

        for key, count
        in duplicate_counter.items()

        if count > 1
    ]


    duplicate_record_count = len(
        duplicate_records
    )


    duplicate_records_absent = (
        duplicate_record_count
        == 0
    )


else:

    duplicate_records = None

    duplicate_record_count = None

    duplicate_records_absent = None


# ------------------------------------------------------------
# E. Value-type counts
# ------------------------------------------------------------

if records_evaluable:

    numeric_value_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and isinstance(
                record.get(
                    "Value"
                ),
                (
                    int,
                    float
                )
            )

            and not isinstance(
                record.get(
                    "Value"
                ),
                bool
            )
        )
    )


    text_value_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and isinstance(
                record.get(
                    "Value"
                ),
                str
            )
        )
    )


    null_value_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Value"
            )
            is None
        )
    )


else:

    numeric_value_count = None

    text_value_count = None

    null_value_count = None


# ------------------------------------------------------------
# F. Topic-level diagnostics
# ------------------------------------------------------------

def find_topic_records(
    topic,
    category=None
):

    if not records_evaluable:

        return []


    return [
        record

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Topic"
            )
            == topic

            and (
                category is None
                or record.get(
                    "Category"
                )
                == category
            )
        )
    ]


def find_unique_topic_record(
    topic,
    category=None
):

    matches = find_topic_records(
        topic,
        category
    )


    if len(
        matches
    ) != 1:

        return None


    return matches[0]


if records_evaluable:

    publication_topic_status = {
        topic: (
            len(
                find_topic_records(
                    topic,
                    "Publication metadata"
                )
            )
            == 1
        )

        for topic
        in PUBLICATION_METADATA_TOPICS
    }


    selected_index_topic_status = {
        topic: (
            len(
                find_topic_records(
                    topic,
                    "Index entry"
                )
            )
            == 1
        )

        for topic
        in INDEX_ENTRY_TOPICS
    }


    statistical_topic_status = {
        topic: (
            len(
                find_topic_records(
                    topic,
                    "Statistical value"
                )
            )
            == 1
        )

        for topic
        in STATISTICAL_TOPICS
    }


    structural_topic_status = {
        topic: (
            len(
                find_topic_records(
                    topic,
                    "Document structure"
                )
            )
            == 1
        )

        for topic
        in STRUCTURAL_TOPICS
    }


    publication_topics_complete = all(
        publication_topic_status.values()
    )


    selected_index_topics_complete = all(
        selected_index_topic_status.values()
    )


    statistical_topics_complete = all(
        statistical_topic_status.values()
    )


    structural_topics_complete = all(
        structural_topic_status.values()
    )


else:

    publication_topic_status = None

    selected_index_topic_status = None

    statistical_topic_status = None

    structural_topic_status = None

    publication_topics_complete = None

    selected_index_topics_complete = None

    statistical_topics_complete = None

    structural_topics_complete = None


# ------------------------------------------------------------
# G. Five fixed Tabela I statistical-value diagnostics
# ------------------------------------------------------------

if records_evaluable:

    statistical_value_status = {}


    for topic, expected_value in (
        EXPECTED_STATISTICAL_VALUES.items()
    ):

        record = find_unique_topic_record(
            topic,
            "Statistical value"
        )


        if record is None:

            statistical_value_status[
                topic
            ] = False

            continue


        observed_value = record.get(
            "Value"
        )


        statistical_value_status[
            topic
        ] = (
            isinstance(
                observed_value,
                (
                    int,
                    float
                )
            )

            and not isinstance(
                observed_value,
                bool
            )

            and math.isclose(
                float(
                    observed_value
                ),
                float(
                    expected_value
                ),
                rel_tol=1e-9,
                abs_tol=1e-9
            )
        )


    all_statistical_values_preserved = all(
        statistical_value_status.values()
    )


else:

    statistical_value_status = None

    all_statistical_values_preserved = None


# ------------------------------------------------------------
# H. Reference-year versus publication-year diagnostic
# ------------------------------------------------------------

if records_evaluable:

    reference_year_record = (
        find_unique_topic_record(
            "Reference year",
            "Publication metadata"
        )
    )


    publication_year_record = (
        find_unique_topic_record(
            "Publication year",
            "Publication metadata"
        )
    )


    year_distinction_preserved = all([
        reference_year_record
        is not None,

        publication_year_record
        is not None,

        (
            reference_year_record.get(
                "Value"
            )
            == 1925

            if reference_year_record
            is not None

            else False
        ),

        (
            publication_year_record.get(
                "Value"
            )
            == 1929

            if publication_year_record
            is not None

            else False
        )
    ])


else:

    year_distinction_preserved = None


# ------------------------------------------------------------
# I. Selected index-entry physical-page diagnostics
# ------------------------------------------------------------

EXPECTED_INDEX_PHYSICAL_PAGES = {
    "Tabela I": "PDF page 3",
    "Tabela II": "PDF page 3",
    "Tabela III": "PDF page 3",
    "Tabela XIV": "PDF page 3",
    "Tabela LVIII": "PDF page 5",
    "Tabela LIX": "PDF page 5"
}


if records_evaluable:

    index_source_location_status = {}


    for topic, expected_page_prefix in (
        EXPECTED_INDEX_PHYSICAL_PAGES.items()
    ):

        record = find_unique_topic_record(
            topic,
            "Index entry"
        )


        source_location = (
            record.get(
                "Source Location"
            )
            if record
            is not None
            else None
        )


        index_source_location_status[
            topic
        ] = (
            isinstance(
                source_location,
                str
            )

            and source_location.startswith(
                expected_page_prefix
            )
        )


    index_source_locations_valid = all(
        index_source_location_status.values()
    )


else:

    index_source_location_status = None

    index_source_locations_valid = None


# ------------------------------------------------------------
# J. Structural-record diagnostics
# ------------------------------------------------------------

if records_evaluable:

    rotated_table_record = (
        find_unique_topic_record(
            "Rotated table",
            "Document structure"
        )
    )


    bilingual_headings_record = (
        find_unique_topic_record(
            "Bilingual headings",
            "Document structure"
        )
    )


    historical_typography_record = (
        find_unique_topic_record(
            "Historical typography",
            "Document structure"
        )
    )


    rotated_table_record_present = (
        rotated_table_record
        is not None
    )


    bilingual_headings_record_present = (
        bilingual_headings_record
        is not None
    )


    historical_typography_record_present = (
        historical_typography_record
        is not None
    )


else:

    rotated_table_record_present = None

    bilingual_headings_record_present = None

    historical_typography_record_present = None


# ------------------------------------------------------------
# Content diagnostics
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        record_count_valid,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records_absent":
        duplicate_records_absent,

    "numeric_value_count":
        numeric_value_count,

    "text_value_count":
        text_value_count,

    "null_value_count":
        null_value_count,

    "publication_topic_status":
        publication_topic_status,

    "publication_topics_complete":
        publication_topics_complete,

    "selected_index_topic_status":
        selected_index_topic_status,

    "selected_index_topics_complete":
        selected_index_topics_complete,

    "statistical_topic_status":
        statistical_topic_status,

    "statistical_topics_complete":
        statistical_topics_complete,

    "structural_topic_status":
        structural_topic_status,

    "structural_topics_complete":
        structural_topics_complete,

    "statistical_value_status":
        statistical_value_status,

    "all_statistical_values_preserved":
        all_statistical_values_preserved,

    "year_distinction_preserved":
        year_distinction_preserved,

    "index_source_location_status":
        index_source_location_status,

    "index_source_locations_valid":
        index_source_locations_valid,

    "rotated_table_record_present":
        rotated_table_record_present,

    "bilingual_headings_record_present":
        bilingual_headings_record_present,

    "historical_typography_record_present":
        historical_typography_record_present
}


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Records with structure issues:",
    records_with_structure_issues
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Records with type issues:",
    records_with_type_issues
)

print(
    "Observed records:",
    observed_record_count
)

print(
    "Record count matches reference:",
    record_count_valid
)

print(
    "Category counts match reference:",
    category_counts_valid
)

print(
    "Missing mandatory values:",
    missing_mandatory_value_count
)

print(
    "Duplicate complete records:",
    duplicate_record_count
)

print(
    "Publication topics complete:",
    publication_topics_complete
)

print(
    "Selected index topics complete:",
    selected_index_topics_complete
)

print(
    "Statistical topics complete:",
    statistical_topics_complete
)

print(
    "Structural topics complete:",
    structural_topics_complete
)

print(
    "All five statistical values preserved:",
    all_statistical_values_preserved
)

print(
    "Reference/publication years distinct:",
    year_distinction_preserved
)

print(
    "Index source locations valid:",
    index_source_locations_valid
)


print(
    "\nObserved category counts:"
)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
    if observed_category_counts
    is not None
    else None
)


print(
    "\nStatistical value diagnostics:"
)

print(
    json.dumps(
        statistical_value_status,
        ensure_ascii=False,
        indent=2
    )
    if statistical_value_status
    is not None
    else None
)

In [ ]:
# ============================================================
# 7. Technical diagnostic summary and experiment metadata
# ============================================================

# ------------------------------------------------------------
# Technical/schema validity only
# ------------------------------------------------------------

STRUCTURAL_CHECKS = {
    "valid_json":
        bool(
            valid_json
        ),

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_present":
        bool(
            document_id_present
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_present":
        bool(
            branch_present
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_present":
        bool(
            records_present
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "record_schema_valid":
        (
            record_schema_valid
            if records_evaluable
            else None
        ),

    "field_types_valid":
        (
            field_types_valid
            if records_evaluable
            else None
        )
}


structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])


# ------------------------------------------------------------
# Structure check artifact
# ------------------------------------------------------------

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "structural_checks":
        STRUCTURAL_CHECKS,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issue_count":
        (
            len(
                field_type_issues
            )
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "missing_mandatory_values":
        (
            missing_mandatory_values
            if records_evaluable
            else None
        ),

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records":
        duplicate_records,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        )
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment metadata
# ------------------------------------------------------------

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_page_count":
            EXPECTED_PAGE_COUNT,

        "observed_page_count":
            PAGE_COUNT,

        "page_count_verified":
            PAGE_COUNT_VALID,

        "meaningful_native_text_layer":
            TEXT_EXTRACTABLE,

        "image_based_source":
            IMAGE_BASED_SOURCE,

        "ocr_dependency_for_text_only_processing":
            OCR_DEPENDENCY_FOR_TEXT_ONLY_PROCESSING,

        "stage1_visual_characteristics":
            STAGE1_VISUAL_CHARACTERISTICS
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "diagnostic_native_text_inspection_applied":
        True,

    "native_text_used_as_model_input":
        False,

    "diagnostic_page_rendering_applied":
        True,

    "rendered_images_used_as_model_input":
        False,

    "ocr_applied_for_model_input":
        False,

    "ocr_transcript_used_as_model_input":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "page_extraction_applied":
        False,

    "page_cropping_applied":
        False,

    "page_rotation_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_reconstruction_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "complete_original_pdf_supplied":
        True,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_expectations_disclosed_to_model":
        False,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED,

    "representation_file":
        REPRESENTATION_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        (
            "JSON object with document_id, "
            "branch and records"
        ),

    "execution_environment":
        "Independent ChatGPT conversation",

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "reference_schema_version":
        "v1",

    "reference_task_version":
        "v1",

    "notebook_version":
        "Branch A D9 v2",

    "notes": (
        "Branch A submits the complete original D9 historical "
        "scanned PDF directly to the model. PyMuPDF native-text "
        "inspection and 150-DPI page rendering are used only for "
        "source-integrity and representation diagnostics. Neither "
        "diagnostic text nor rendered diagnostic images are used as "
        "the model representation. No OCR transcript, PDF-to-text "
        "conversion, page rotation, page extraction, cropping, "
        "structural conversion, table reconstruction, normalisation, "
        "semantic rewriting, unit conversion or manual correction is "
        "applied before extraction. Stage 1 reference values, expected "
        "record count and expected category distribution are not "
        "supplied to the model. Content-level validation is performed separately in "
        "Validation A — D9."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment summary
# ------------------------------------------------------------

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        INPUT_INTEGRITY_PASSED,

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "image_based_source":
        IMAGE_BASED_SOURCE,

    "ocr_dependency_for_text_only_processing":
        OCR_DEPENDENCY_FOR_TEXT_ONLY_PROCESSING,

    "ocr_applied_for_model_input":
        False,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "numeric_value_count":
        numeric_value_count,

    "text_value_count":
        text_value_count,

    "null_value_count":
        null_value_count,

    "publication_topics_complete":
        publication_topics_complete,

    "selected_index_topics_complete":
        selected_index_topics_complete,

    "statistical_topics_complete":
        statistical_topics_complete,

    "structural_topics_complete":
        structural_topics_complete,

    "all_statistical_values_preserved":
        all_statistical_values_preserved,

    "year_distinction_preserved":
        year_distinction_preserved,

    "index_source_locations_valid":
        index_source_locations_valid,

    "rotated_table_record_present":
        rotated_table_record_present,

    "bilingual_headings_record_present":
        bilingual_headings_record_present,

    "historical_typography_record_present":
        historical_typography_record_present,

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        parsed_extraction_created,

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, representation "
        "characterisation, D9 Branch A direct scanned-PDF execution "
        "preservation, technical/schema checks and document-specific "
        "content diagnostics only. Formal agreement with the fixed "
        "Stage 1 reference dataset is evaluated separately in "
        "Validation A — D9."
    )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print(
    "Structural checks:"
)

print(
    json.dumps(
        STRUCTURAL_CHECKS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nContent diagnostics:"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nStructurally evaluable:",
    structurally_evaluable
)


print(
    "\nExperiment summary:"
)

print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\n" + "=" * 60
)

print(
    "D9 Branch A experiment completed"
)

print(
    "=" * 60
)


print(
    "Input integrity passed        :",
    INPUT_INTEGRITY_PASSED
)

print(
    "Image-based source            :",
    IMAGE_BASED_SOURCE
)

print(
    "OCR used for model input      : False"
)

print(
    "Raw response preserved        :",
    RAW_RESPONSE_PATH.exists()
)

print(
    "Valid JSON                    :",
    valid_json
)

print(
    "Records evaluable             :",
    records_evaluable
)

print(
    "Expected records              :",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records              :",
    (
        observed_record_count
        if observed_record_count
        is not None
        else "Not evaluable"
    )
)

print(
    "Record count matches          :",
    record_count_valid
)

print(
    "Category counts match         :",
    category_counts_valid
)

print(
    "Record schema valid           :",
    record_schema_valid
)

print(
    "Field types valid             :",
    field_types_valid
)

print(
    "Structurally evaluable       :"
    structurally_evaluable
)

print(
    "Content validation performed  : False"
)

print(
    "Next step                     : Validation A — D9"
)


# ------------------------------------------------------------
# Output existence checks
# ------------------------------------------------------------

required_output_paths = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if (
    parsed_extraction_created
    and PARSED_EXTRACTION_PATH.exists()
):

    required_output_paths.append(
        PARSED_EXTRACTION_PATH
    )


missing_output_files = [
    path.name

    for path
    in required_output_paths

    if not path.exists()
]


if missing_output_files:

    raise AssertionError(
        "Missing output files: "
        f"{missing_output_files}"
    )


print(
    "\nGenerated D9 Branch A files:\n"
)


for path in required_output_paths:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )